# Setup Lakebase App Schema

Creates the `app` schema in the Lakebase database and grants the app's
auto-provisioned SPN full privileges on it. Run this before the app starts
so migrations can create tables in the `app` schema.

**Parameters:**
- `principal` — App SPN client_id (also the PG role name in Lakebase)
- `lakebase_project_id` — Lakebase project ID
- `lakebase_database_id` — Lakebase database resource ID

Triggered by deploy.sh via the `grant_secrets_acl` bundle job.

In [ ]:
%pip install --upgrade databricks-sdk "psycopg[binary]>=3.1.0"

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("principal", "", "App SPN Client ID (PG role)")
dbutils.widgets.text("lakebase_project_id", "", "Lakebase Project ID")
dbutils.widgets.text("lakebase_database_id", "", "Lakebase Database ID")

In [ ]:
principal = dbutils.widgets.get("principal")
lakebase_project_id = dbutils.widgets.get("lakebase_project_id")
lakebase_database_id = dbutils.widgets.get("lakebase_database_id")

assert principal, "principal is required"
assert lakebase_project_id, "lakebase_project_id is required"
assert lakebase_database_id, "lakebase_database_id is required"

# Construct the Lakebase resource paths
branch_path = f"projects/{lakebase_project_id}/branches/production"
database_path = f"{branch_path}/databases/{lakebase_database_id}"

print(f"  Principal (PG role): {principal}")
print(f"  Lakebase project:   {lakebase_project_id}")
print(f"  Database:           {lakebase_database_id}")
print(f"  Branch path:        {branch_path}")

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Discover the endpoint for this branch
endpoints = w.postgres.list_endpoints(branch_path)
endpoint = next(iter(endpoints), None)

if endpoint is None:
    raise RuntimeError(
        f"No endpoint found for branch '{branch_path}'. "
        f"The Lakebase project may still be initializing."
    )

endpoint_name = endpoint.name
pg_host = endpoint.hostname
pg_port = endpoint.port or 5432

print(f"  Endpoint:  {endpoint_name}")
print(f"  Host:      {pg_host}")
print(f"  Port:      {pg_port}")

In [ ]:
# Generate a short-lived database credential for the current user
credential = w.postgres.generate_database_credential(endpoint=endpoint_name)

# The current user's PG role (should be the notebook runner — an admin)
pg_user = credential.username
pg_token = credential.token
pg_database = "databricks_postgres"  # Default Lakebase database name

print(f"  Connected as PG user: {pg_user}")
print(f"  Database: {pg_database}")

In [ ]:
import psycopg

conninfo = (
    f"host={pg_host} port={pg_port} dbname={pg_database} "
    f"user={pg_user} password={pg_token} sslmode=require"
)

with psycopg.connect(conninfo, autocommit=True) as conn:
    with conn.cursor() as cur:
        # Verify connection
        cur.execute("SELECT current_user, current_database()")
        row = cur.fetchone()
        print(f"  Connected: user={row[0]}, db={row[1]}")

        # 1. Create the app schema
        cur.execute("CREATE SCHEMA IF NOT EXISTS app")
        print("  ✓ Schema 'app' exists")

        # 2. Grant USAGE + CREATE on the schema to the app SPN role
        # In Lakebase, PG role name = SPN client_id (UUID)
        cur.execute(
            f'GRANT ALL ON SCHEMA app TO "{principal}"'
        )
        print(f"  ✓ Granted ALL ON SCHEMA app TO {principal[:8]}...")

        # 3. Set default privileges so future tables created BY the admin
        #    are also accessible to the app role
        cur.execute(
            f'ALTER DEFAULT PRIVILEGES IN SCHEMA app '
            f'GRANT ALL ON TABLES TO "{principal}"'
        )
        print(f"  ✓ Default privileges set for future tables")

        # 4. Verify: list schemas the app role can access
        cur.execute(
            "SELECT schema_name FROM information_schema.schemata ORDER BY schema_name"
        )
        schemas = [r[0] for r in cur.fetchall()]
        print(f"  Schemas in database: {schemas}")

print("\n  Done. App SPN can now CREATE/ALTER/DROP in the 'app' schema.")